In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk

import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128

from IPython.display import HTML
from astropy.time import Time
import astropy.units as u
from tqdm import tqdm
from matplotlib import animation
from matplotlib import colors
from matplotlib import patches
from datetime import datetime

from tess_asteroids import MovingTPF
from tess_asteroids.utils import animate_cube

DATADIR = "/Users/jimartin/Work/TESS/tess-asteroid-ml/data"

# We open the track file

In [ ]:
# Initialise
target = MovingTPF.from_name("2017 K2", sector=26)

In [ ]:
target.get_data(shape=(21,21))
target.time.shape, target.shape

In [ ]:
target.reshape_data()
target.background_correction(method="rolling", nframes=51)
target.create_pixel_quality()

In [ ]:
# del target.prf_model
target.create_aperture(method="prf", threshold=0.01)

In [ ]:
target.shape, target.prf_model.shape

In [ ]:
anim_fname = f"tess_{target.target.replace(' ', '')}_s{target.sector:04}-{target.camera}-{target.ccd}_track{1:02}_anim_{'prf'}.gif"

target.animate_tpf(interval=100, cnorm=False, step=2)

In [ ]:
anim_fname = f"tess_{target.target.replace(' ', '')}_s{sector:04}-{camera}-{ccd}_track{trackno:02}_anim_prf_model.gif"

anim = animate_cube(
    target.bg, 
    aperture_mask=target.aperture_mask, 
    corner=target.corner, 
    ephemeris=target.ephemeris,
    cadenceno=target.cadence_number,
    time=target.time,
    interval=200, 
    repeat_delay=1000,
    cnorm=True,
    step=3,
    suptitle="RAW Flux",
)
# anim.save(anim_fname, writer="pillow")

HTML(anim.to_jshtml())

In [ ]:
bar = plt.imshow(np.nanmedian(target.flux, axis=0), origin="lower")
plt.title(f"Median stack {len(target.time)} frames")
plt.xlabel("Pixel Column")
plt.ylabel("Pixel Row")
plt.colorbar(bar, label="Flux [e-/s]")
plt.show()

In [ ]:
bad_bits = [1, 3]

In [ ]:
target.create_aperture(method="threshold", threshold=2)

value = 0
for bit in bad_bits:
    value += 2 ** (bit - 1)

mask = []
tap_flux = []
tap_flux_err = []
tap_bg = []
tap_bg_err = []

for t in range(len(target.time)):
    # Combine aperture mask with masking of bad bits.
    mask.append(
        np.logical_and(
            target.aperture_mask[t], target.pixel_quality[t] & value == 0
        )
    )

    # Compute flux and bg flux inside aperture (sum of all pixels).
    # (If no pixels in mask, these values will be nan.)
    tap_flux.append(np.nansum(target.corr_flux[t][mask[-1]]))
    tap_flux_err.append(np.sqrt(np.nansum(target.corr_flux_err[t][mask[-1]] ** 2)))
    tap_bg.append(np.nansum(target.bg[t][mask[-1]]))
    tap_bg_err.append(np.sqrt(np.nansum(target.bg_err[t][mask[-1]] ** 2)))

    # If all pixels in aperture have nan value, propagate nan:
    if np.isnan(target.corr_flux[t][mask[-1]]).all():
        tap_flux[-1] = np.nan
    if np.isnan(target.corr_flux_err[t][mask[-1]]).all():
        tap_flux_err[-1] = np.nan
    if np.isnan(target.bg[t][mask[-1]]).all():
        tap_bg[-1] = np.nan
    if np.isnan(target.bg_err[t][mask[-1]]).all():
        tap_bg_err[-1] = np.nan

tap_flux = np.asarray(tap_flux)
tap_flux_err = np.asarray(tap_flux_err)
tap_bg = np.asarray(tap_bg)
tap_bg_err = np.asarray(tap_bg_err)

tap_mag = -2.5 * np.log10(tap_flux) + 20.44
tap_mag_err = np.sqrt((tap_flux_err/(tap_flux * np.log(10))) ** 2 + 0.05 ** 2)

In [ ]:
target.create_aperture(method="prf", threshold=0.01)

value = 0
for bit in bad_bits:
    value += 2 ** (bit - 1)

mask = []
pap_flux = []
pap_flux_err = []
pap_bg = []
pap_bg_err = []

for t in range(len(target.time)):
    # Combine aperture mask with masking of bad bits.
    mask.append(
        np.logical_and(
            target.aperture_mask[t], target.pixel_quality[t] & value == 0
        )
    )

    # Compute flux and bg flux inside aperture (sum of all pixels).
    # (If no pixels in mask, these values will be nan.)
    pap_flux.append(np.nansum(target.corr_flux[t][mask[-1]]))
    pap_flux_err.append(np.sqrt(np.nansum(target.corr_flux_err[t][mask[-1]] ** 2)))
    pap_bg.append(np.nansum(target.bg[t][mask[-1]]))
    pap_bg_err.append(np.sqrt(np.nansum(target.bg_err[t][mask[-1]] ** 2)))

    # If all pixels in aperture have nan value, propagate nan:
    if np.isnan(target.corr_flux[t][mask[-1]]).all():
        pap_flux[-1] = np.nan
    if np.isnan(target.corr_flux_err[t][mask[-1]]).all():
        pap_flux_err[-1] = np.nan
    if np.isnan(target.bg[t][mask[-1]]).all():
        pap_bg[-1] = np.nan
    if np.isnan(target.bg_err[t][mask[-1]]).all():
        pap_bg_err[-1] = np.nan

pap_flux = np.asarray(pap_flux)
pap_flux_err = np.asarray(pap_flux_err)
pap_bg = np.asarray(pap_bg)
pap_bg_err = np.asarray(pap_bg_err)

pap_mag = -2.5 * np.log10(pap_flux) + 20.44
pap_mag_err = np.sqrt((pap_flux_err/(pap_flux * np.log(10))) ** 2 + 0.05 ** 2)

In [ ]:
plt.figure(figsize=(12,3))
# plt.errorbar(target.time, pap_mag, yerr=0.1, label="PRF")
plt.errorbar(target.time, tap_flux, yerr=0, label="Thr")
plt.errorbar(target.time, tap_bg, yerr=0, label="Bkg")
plt.xlabel("Time [JD - 2457000]")
plt.ylabel("Fux [e-/s]")
plt.legend(loc="upper right")
# plt.gca().invert_yaxis()
plt.yscale("log")
plt.show()

In [ ]:
plt.figure(figsize=(12,3))
# plt.errorbar(target.time, pap_mag, yerr=0.1, label="PRF")
plt.errorbar(target.time, tap_mag, yerr=tap_mag_err, label="Thr")
plt.xlabel("Time [JD - 2457000]")
plt.ylabel("TESS mag")
plt.legend(loc="upper right")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
import lightkurve as lk

In [ ]:
lc = lk.LightCurve(time=target.time, flux=tap_flux, flux_err=tap_flux_err)

In [ ]:
per = lc.to_periodogram(method="lombscargle", minimum_frequency=1/5, maximum_frequency=1/0.1)

In [ ]:
per.plot()

In [ ]:
per.show_properties()

In [ ]:
lc.fold(period=per.period_at_max_power * 1).plot()

In [ ]:
try:
    target.cube.wcss
    print("Using every WCSs")
    radec = []
    for i, cad in enumerate(target.cadence_number):
        idx = np.where(cad == target.cube.cadence_number)[0][0]
        if target.cube.wcss[idx] is not None:
            radec.append(
                target.cube.wcss[idx].all_pix2world(
                    target.ephem.loc[i, ["column", "row"]].values[None, :], 0
                )[0]
            )
        else:
            radec.append(
                target.cube.wcs.all_pix2world(
                    target.ephem.loc[i, ["column", "row"]].values[None, :], 0
                )[0]
            )

    ra, dec = np.asarray(radec).T
except TypeError:
    print("Using average WCS")
    ra, dec = target.cube.wcs.all_pix2world(
        target.ephem.loc[:, ["column", "row"]].values, 0
    ).T

In [ ]:
np.isin(target.cadence_number, target.cube.cadence_number).shape

In [ ]:
len(target.cube.wcss)

In [ ]:
ra, dec = target.cube.wcs.all_pix2world(target.ephem.loc[:, ["column", "row"]].values, 0).T

In [ ]:
target.cube.timecorr[np.isin(target.cube.cadence_number, target.cadence_number)]

In [ ]:
target.cube.wcs.to_header()

In [ ]:
lc = pd.DataFrame(
    np.array([
        target.cadence_number,
        target.time + 2457000,
        target.time + 2457000 + target.cube.timecorr[np.isin(target.cube.cadence_number, target.cadence_number)],
        ra,
        dec,
        pap_flux,
        pap_flux_err,
        pap_mag,
        pap_mag_err,
        tap_flux,
        tap_flux_err,
        tap_mag,
        tap_mag_err,
    ]).T,
    columns= ["cadence", "time_bjd", "time_utc", "ra", "dec", "flux", "flux_err", "mag", "mag_err", "thr_flux", "thr_flux_err", "thr_mag", "thr_mag_err"]
)
lc

In [ ]:
lc_fname = f"tess_{target.target.replace(' ', '')}_s{sector:04}-{camera}-{ccd}_lc{trackno:02}.csv"
print(lc_fname)

lc.to_csv(f"{DATADIR}/asteroid_lcs/{lc_fname}")

## Open all LC files for the target

In [ ]:
from glob import glob

In [ ]:
lc_files = glob(f"{DATADIR}/asteroid_lcs/tess_2024Brian_*.csv")
lc_files

In [ ]:
lcs = pd.concat([pd.read_csv(x, index_col=0) for x in lc_files]).reset_index(drop=True)
lcs

In [ ]:
lcs = lcs.sort_values("cadence")

In [ ]:
plt.figure(figsize=(12,3))

plt.title(f"{target.target} Sector {sector} Camera {camera}")
plt.errorbar(lcs.time - 2457000, lcs.mag, yerr=lcs.mag_err, label="PRF", fmt='o', ms=2)
# plt.errorbar(lcs.time - 2457000, lcs.thr_mag, yerr=lcs.thr_mag_err, label="Thr", fmt='o', ms=2)
plt.xlabel("Time [JD - 2457000]")
plt.ylabel("TESS mag")
plt.gca().invert_yaxis()
plt.legend(loc="upper right")
# plt.ylim(18, 14)

lc_fname = f"tess_{target.target.replace(' ', '')}_s{sector:04}-{camera}_lc-stitched.png"
plt.savefig(f"{DATADIR}/asteroid_lcs/{lc_fname}", format="png", bbox_inches="tight")

In [ ]:
plt.plot(lcs.ra, lcs.dec)
plt.xlabel("R.A [deg]")
plt.ylabel("Dec [deg]")

In [ ]:
from astroquery.jplhorizons import Horizons

In [ ]:
TIME = Time(target.time + 2457000, format="jd", scale="tdb")
TIME

In [ ]:
TIME[0].jd 

In [ ]:
obj = Horizons(id='-95', location='500', epochs=TIME.jd)

In [ ]:
obj.id

In [ ]:
obj.vectors()

In [ ]:
obj.vectors()["x", "y", "z"]

In [ ]:
x = obj.vectors()["x"].to("km").value
y = obj.vectors()["y"].to("km").value
z = obj.vectors()["z"].to("km").value

In [ ]:
x, y, z

In [ ]:
neo_tpf, nep_ephem = MovingTPF.from_name("2018 TF3", sector=4, camera=4, ccd=2, time_step=0.0001)

In [ ]:
nep_ephem

In [ ]:
neo_tpf.get_data(shape=(101, 101))